# LeetCode #1311: Get Watched Videos by Your Friends

https://leetcode.com/problems/get-watched-videos-by-your-friends/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: BFS Level-by-Level ★** | $O(n + E + V \log V)$ | $O(n + V)$ |

---

## Understanding the Methods

### Brute Force
Compute distances from node 0 to every other node naively, collect all friends at distance `k`, tally their videos, and sort. Repeated distance computation is wasteful.

### Optimal: BFS Level-by-Level ★
BFS from person 0 stops exactly at level `k`. Collect every video watched by a level-`k` friend, count frequencies, then sort: primary key = frequency (ascending), secondary key = title (lexicographic).

**Constraints:**
* $2 \le n \le 100$
* $1 \le watchedVideos[i].length \le 100$
* $0 \le k < n$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
using System.Linq;

public class Solution {
    public IList<string> WatchedVideosByFriends(
        IList<IList<string>> watchedVideos, int[][] friends, int id, int level) {

        // BFS from id, stopping once we reach the target level
        var visited = new bool[friends.Length];
        visited[id] = true;
        var q = new Queue<int>();
        q.Enqueue(id);

        int curLevel = 0;
        while (curLevel < level) {
            int sz = q.Count;
            curLevel++;
            for (int i = 0; i < sz; i++) {
                int person = q.Dequeue();
                foreach (int f in friends[person]) {
                    if (!visited[f]) { visited[f] = true; q.Enqueue(f); }
                }
            }
        }

        // Count videos watched by all friends at exactly level k
        var freq = new Dictionary<string, int>();
        while (q.Count > 0) {
            int person = q.Dequeue();
            foreach (var v in watchedVideos[person]) {
                freq.TryGetValue(v, out int cnt);
                freq[v] = cnt + 1;
            }
        }

        // Sort by frequency ascending, then alphabetically
        return freq.OrderBy(kv => kv.Value).ThenBy(kv => kv.Key)
                   .Select(kv => kv.Key).ToList();
    }
}

### Python

In [ ]:
from collections import deque, defaultdict

class Solution:
    def watchedVideosByFriends(self, watchedVideos: list[list[str]], friends: list[list[int]], id: int, level: int) -> list[str]:
        # BFS to reach exactly the k-th degree friends
        visited = {id}
        q = deque([id])
        for _ in range(level):
            for _ in range(len(q)):
                person = q.popleft()
                for f in friends[person]:
                    if f not in visited:
                        visited.add(f)
                        q.append(f)

        # Tally video frequencies among the level-k friends still in the queue
        freq: dict[str, int] = defaultdict(int)
        for person in q:
            for video in watchedVideos[person]:
                freq[video] += 1

        # Primary sort: frequency ascending; secondary: lexicographic
        return sorted(freq, key=lambda v: (freq[v], v))

### Go

In [ ]:
import "sort"

func watchedVideosByFriends(watchedVideos [][]string, friends [][]int, id int, level int) []string {
	visited := map[int]bool{id: true}
	q := []int{id}

	// Advance BFS by exactly `level` hops
	for l := 0; l < level; l++ {
		next := []int{}
		for _, person := range q {
			for _, f := range friends[person] {
				if !visited[f] {
					visited[f] = true
					next = append(next, f)
				}
			}
		}
		q = next
	}

	// Count video frequencies among level-k friends
	freq := map[string]int{}
	for _, person := range q {
		for _, v := range watchedVideos[person] {
			freq[v]++
		}
	}

	videos := make([]string, 0, len(freq))
	for v := range freq { videos = append(videos, v) }
	// Sort by frequency then title
	sort.Slice(videos, func(i, j int) bool {
		fi, fj := freq[videos[i]], freq[videos[j]]
		if fi != fj { return fi < fj }
		return videos[i] < videos[j]
	})
	return videos
}

### Rust

In [ ]:
use std::collections::{HashMap, HashSet, VecDeque};

impl Solution {
    pub fn watched_videos_by_friends(
        watched_videos: Vec<Vec<String>>,
        friends: Vec<Vec<i32>>,
        id: i32,
        level: i32,
    ) -> Vec<String> {
        let id = id as usize;
        let mut visited: HashSet<usize> = HashSet::from([id]);
        let mut q: VecDeque<usize> = VecDeque::from([id]);

        // BFS: advance exactly `level` layers
        for _ in 0..level {
            let sz = q.len();
            for _ in 0..sz {
                let person = q.pop_front().unwrap();
                for &f in &friends[person] {
                    let f = f as usize;
                    if visited.insert(f) { q.push_back(f); }
                }
            }
        }

        // Tally frequencies from level-k friends
        let mut freq: HashMap<&str, i32> = HashMap::new();
        for person in &q {
            for v in &watched_videos[*person] {
                *freq.entry(v).or_default() += 1;
            }
        }

        // Sort by frequency asc, then lexicographically
        let mut videos: Vec<&str> = freq.keys().copied().collect();
        videos.sort_by(|a, b| freq[a].cmp(&freq[b]).then(a.cmp(b)));
        videos.into_iter().map(|s| s.to_owned()).collect()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `watchedVideos=[["A","B"],["C"],["B","C"],["D"]], friends=[[1,2],[0,3],[0,3],[1,2]], id=0, level=1`
Level-1 friends of 0 are {1, 2}. Videos: A×1, B×2, C×2. Sorted by frequency then alpha: `["A","B","C"]`.

### 2. Slightly Complex
**Input:** Same graph, `id=0, level=2`
Level-2 friends: {3} (only unvisited from level-1's neighbours). Videos: ["D"]. Result: `["D"]`.

### 3. Edge Case: Time Factor
**Input:** $n=100$, each person friends with all others; `level=1`.
BFS processes $n-1$ friends in one step. Video counting iterates over all $100$ video lists. Total: $O(n + V)$ where $V$ = total videos.

### 4. Edge Case: Space Factor
**Input:** All 100 people are at level-1 from the source; each watches 100 unique videos.
`freq` map holds up to $10000$ video strings — $O(n \cdot L)$ space.

### 5. Almost-Impossible but Plausible
**Input:** Friendship graph is a long path: $0-1-2-\cdots-99$; `id=0, level=50`.
BFS advances exactly 50 hops in 50 iterations, visiting only person 50. Only that person's videos are counted, avoiding any quadratic blowup despite the $O(n^2)$ naive distance approach.